In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from sklearn.metrics import (
confusion_matrix,
ConfusionMatrixDisplay,
classification_report,
accuracy_score
)
wineData = load_wine()
X = pd.DataFrame(
wineData.data,
columns=wineData.feature_names
)
y = pd.Series(wineData.target)
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print(X.head())
print(y.value_counts().sort_index())

Shape of X: (178, 13)
Shape of y: (178,)
   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80   
1    13.20        1.78  2.14               11.2      100.0           2.65   
2    13.16        2.36  2.67               18.6      101.0           2.80   
3    14.37        1.95  2.50               16.8      113.0           3.85   
4    13.24        2.59  2.87               21.0      118.0           2.80   

   flavanoids  nonflavanoid_phenols  proanthocyanins  color_intensity   hue  \
0        3.06                  0.28             2.29             5.64  1.04   
1        2.76                  0.26             1.28             4.38  1.05   
2        3.24                  0.30             2.81             5.68  1.03   
3        3.49                  0.24             2.18             7.80  0.86   
4        2.69                  0.39             1.82             4.32  1.04   

   od280/od315_of_dil

1. How many observations are in the dataset? - 178 rows
2. How many input features does each observation have? - 13 columns
3. How many classes are there? - 0: all numerical columns
4. Is this a regression or classification problem? Why? - Regression because we are working with numerical values to predict a numerical answer

In [ ]:
# First: separate training data from the remaining data
X_train, X_temp, y_train, y_temp = train_test_split(
X, y,
test_size=0.40,
random_state=42,
stratify=y
)
# Second: split the remaining data into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
X_temp, y_temp,
test_size=0.50,
random_state=42,
stratify=y_temp
)
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (106, 13)
Validation: (36, 13)
Testing: (36, 13)


In [ ]:
scaler = StandardScaler()

# Fit only on training data
X_train = scaler.fit_transform(X_train)

# Transform validation and test data using the same scaler
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

#Convert the data into PyTorch tensors:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_val = torch.tensor(y_val.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

1. Why do we standardize the features? - To make it uniform to prevent skewing the data too far
2. Why do we fit the scaler using only the training data? - The test data is the data that is used to predict the target value so scaling on the testing data would cause the target value to be leaked and therefore wouldn't be predicting, only reading the actual value
3. What is the purpose of the validation dataset? - To evaluate, tune and select the model that will be used
4. Why should we keep the test dataset separate? - The test data is the set which holds the target value so keeping it separate prevents the model from just reading that target value


In [5]:
hidden_sizes = []
layers = []
input_size = 13 # set the input size according to your dataset
output_size = 3 # set according to the dataset
for hidden_size in hidden_sizes:
  layers.append(
  nn.Linear(input_size, hidden_size)
  )
  layers.append(nn.Tanh())
  input_size = hidden_size
# One output neuron because this is regression
layers.append(nn.Linear(input_size, output_size))
network = nn.Sequential(*layers)

1. Why do we need an activation function? - It introduces nonlinearity to the network in order for it to learn patterns that may be complex because real world data is hardly ever linear
2. How to calculate the number of input neurons in the first layer? - The number of input neurons is how many features of the dataset